[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/example4_conley_morse.ipynb)

# Example 4. Periodic-orbit recovery

The repressilator of Example 3, fixed to DSGRN node **13**, whose Morse graph is a single
**FC** ("full cycle") node, a periodic orbit. The notebook tests recovery of the orbit itself,
not only region membership, and compares pipeline variants:

- the parameter sample is chosen so each threshold sits near the middle of its `[L, U]`
  interval, keeping the steep right-hand side away from its most sensitive configuration;
- each recovered parameter set is scored on two criteria, region membership (`in_region`) and
  whether the smooth model still oscillates (`oscillates`);
- two pipeline variants are evaluated: denoising before gradient-matching, and regenerating the
  data at higher steepness.

The region test uses the explicit inequalities (no DSGRN install); an optional final section
installs DSGRN and computes the Morse graph directly.

In [ ]:
# Colab already has numpy/scipy/matplotlib/torch, so this cell is a no-op there.
# Uncomment if you are running somewhere that is missing a package.
# !pip -q install numpy scipy matplotlib torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy.signal import savgol_filter
import torch, torch.nn as nn
torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=3, suppress=True)

### Hill production function

Each gene is repressed by the previous one, so only the decreasing Hill function appears.
Degradation `gamma` and steepness `d` are fixed; only the nine `(L, U, theta)` are learned.

In [ ]:
GAMMA = 1.0
DATA_D = 12.0    # Hill steepness used to GENERATE the data (held fixed in recovery too)

def hill_rep(x, L, U, th, d):   # decreasing: high U -> low L as x rises
    xd = np.power(np.maximum(x, 1e-12), d); td = np.power(th, d)
    return L + (U - L) * td / (td + xd)

def hill_rep_t(x, L, U, th, d): # torch version, for the PINN physics loss
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * td / (td + xd)

## 1. Model and DSGRN region (node 13)

$$\dot{x}=-x+f^-(z),\quad \dot{y}=-y+f^-(x),\quad \dot{z}=-z+f^-(y).$$

Each production range straddles the threshold it represses, `L < theta < U` around the cycle;
`in_region` below combines these straddle inequalities with positivity of each `(L, U)` range.
The parameters in `P` are a fixed sample from node 13, chosen so each threshold sits near the
middle of its `[L, U]` interval (see the printout).

In [ ]:
# A numerically clean sample from DSGRN parameter node 13 (repressilator FC region).
# Keys: 31 = z->x edge, 12 = x->y edge, 23 = y->z edge; th** are the DSGRN thresholds.
P = dict(L31=0.6688, U31=1.3730, th31=1.3115,
         L12=0.5570, U12=1.4329, th12=1.0269,
         L23=0.4150, U23=1.7400, th23=1.0411)

def rhs(t, s, p, d=DATA_D, g=GAMMA):
    x, y, z = s
    return [-g*x + hill_rep(z, p['L31'], p['U31'], p['th31'], d),
            -g*y + hill_rep(x, p['L12'], p['U12'], p['th12'], d),
            -g*z + hill_rep(y, p['L23'], p['U23'], p['th23'], d)]

def in_region(p):   # DSGRN node 13: each threshold sandwiched by its regulator's (L, U)
    return (p['L31'] < p['th12'] < p['U31'] and
            p['L12'] < p['th23'] < p['U12'] and
            p['L23'] < p['th31'] < p['U23'] and
            0 < p['L31'] < p['U31'] and 0 < p['L12'] < p['U12'] and 0 < p['L23'] < p['U23'])

print('true parameters lie in node 13?', in_region(P))
for L, U, th, name in [(P['L31'],P['U31'],P['th12'],'x'), (P['L12'],P['U12'],P['th23'],'y'),
                       (P['L23'],P['U23'],P['th31'],'z')]:
    print(f'  {name}: threshold sits at {(th-L)/(U-L):.0%} of [L,U]=[{L:.2f},{U:.2f}]')

### Threshold centering

DSGRN fixes only the ordering `L < theta < U`. A draw with `theta` close to `L` or `U` puts the
steep Hill transition near the edge of the orbit's range, where the right-hand side swings
between `L` and `U` over a narrow state window: `solve_ivp` slows and the gradient-matching
residual becomes ill-conditioned. The selected sample places each threshold at 50-70% of its
interval.

In [ ]:
def simulate(p, x0, T, n):
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(p,), rtol=1e-9, atol=1e-11)
    return t, sol.y.T

def add_noise(y, ub_frac, scale, rng):
    return np.clip(y + rng.uniform(-ub_frac*scale, ub_frac*scale, size=y.shape), 0.0, None)

def orbit_amplitude(p, d=DATA_D, T=200.0):
    # simulate long, peak-to-trough over the last 50 time units: >0 means a limit cycle
    sol = solve_ivp(rhs, (0.0, T), [0.5, 0.5, 0.3], t_eval=np.linspace(T-50, T, 400),
                    args=(p, d), rtol=1e-9, atol=1e-11)
    return float((sol.y.max(1) - sol.y.min(1)).max())

def oscillates(p, d=DATA_D):
    return orbit_amplitude(p, d) > 0.05

## 2. Synthetic data

Six initial conditions integrated over a long horizon (a limit cycle), then corrupted with
bounded noise.

In [ ]:
T, n = 40.0, 400
x0s = [[1.,.5,.3], [.3,1.,.5], [.5,.3,1.], [.8,.8,.2], [.2,.6,.9], [.9,.2,.6]]
ts, xs = [], []
for x0 in x0s:
    t, y = simulate(P, x0, T, n); ts.append(t); xs.append(y)

gap = min(abs(P['L31']-P['U31']), abs(P['L12']-P['U12']), abs(P['L23']-P['U23']))
rng = np.random.default_rng(0)
xs_noisy = [add_noise(y, 0.25, gap, rng) for y in xs]

print('true model oscillates?', oscillates(P), ' amplitude =', round(orbit_amplitude(P), 3))

fig = plt.figure(figsize=(11, 3.4))
ax0 = fig.add_subplot(1, 2, 1)
ax0.plot(ts[0], xs[0][:,0], label='$x$'); ax0.plot(ts[0], xs[0][:,1], label='$y$')
ax0.plot(ts[0], xs[0][:,2], label='$z$')
ax0.plot(ts[0], xs_noisy[0][:,0], '.', ms=3, color='C0', alpha=.4)
ax0.set_xlabel('t'); ax0.legend(); ax0.set_title('Repressilator (one trajectory, 25% noise)')
axp = fig.add_subplot(1, 2, 2, projection='3d')
for y in xs: axp.plot(y[:,0], y[:,1], y[:,2], lw=.8, alpha=.7)
axp.set_xlabel('x'); axp.set_ylabel('y'); axp.set_zlabel('z')
axp.set_title('All trajectories wind onto one loop')
plt.tight_layout(); plt.show()

## 3. Least squares: region and oscillation tests

Gradient-matching least squares over the nine `(L, U, theta)` (with `d`, `gamma` fixed). Each
recovered set is scored on both tests, `in_region` and `oscillates`.

In [ ]:
KEYS = ['L31','U31','th31','L12','U12','th12','L23','U23','th23']

def ls_recover(ts, xs, d=DATA_D, g=GAMMA, smooth=False):
    if smooth:   # tweak: denoise each coordinate before differentiating
        xs = [np.column_stack([savgol_filter(y[:,k], 21, 3) for k in range(3)]) for y in xs]
    X  = np.vstack(xs)
    DX = np.vstack([np.gradient(y, t, axis=0) for t, y in zip(ts, xs)])
    def resid(zv):
        p = dict(zip(KEYS, zv))
        f1 = -g*X[:,0] + hill_rep(X[:,2], p['L31'],p['U31'],p['th31'], d)
        f2 = -g*X[:,1] + hill_rep(X[:,0], p['L12'],p['U12'],p['th12'], d)
        f3 = -g*X[:,2] + hill_rep(X[:,1], p['L23'],p['U23'],p['th23'], d)
        return np.concatenate([DX[:,0]-f1, DX[:,1]-f2, DX[:,2]-f3])
    z0 = np.array([0.5,1.5,1.0]*3)   # neutral guess, not the true values
    s = least_squares(resid, z0, bounds=(0, 10), max_nfev=20000)
    return dict(zip(KEYS, s.x))

p_ls = ls_recover(ts, xs_noisy)
print('LS estimate @25% noise:', {k: round(p_ls[k],2) for k in KEYS})
print('  satisfies DSGRN inequalities (in_region):', in_region(p_ls))
print('  learned model oscillates (topology):     ', oscillates(p_ls))

In [ ]:
# Sweep noise; separate the two success criteria.  15 trials each.
def sweep(smooth, data_ts, data_xs, scale, d=DATA_D, label=''):
    print(f'{label}')
    print(f'{"noise":>6} | {"region":>8} | {"topology":>9}')
    for ub in [0.0, 0.1, 0.25, 0.5]:
        r = np.random.default_rng(11); reg = osc = 0
        for _ in range(15):
            xs_n = [add_noise(y, ub, scale, r) for y in data_xs]
            p = ls_recover(data_ts, xs_n, d=d, smooth=smooth)
            reg += in_region(p); osc += oscillates(p, d)
        print(f'{ub:>6} | {reg:>6}/15 | {osc:>7}/15')

sweep(False, ts, xs, gap, label='NAIVE  (raw gradients, d=12 data)')

### Variant 1: denoise before differentiating

A Savitzky-Golay smooth is applied before the finite-difference gradients (same `d=12` data) to
reduce the noise amplified by differentiation.

In [ ]:
sweep(True, ts, xs, gap, label='SMOOTHED  (Savitzky-Golay then gradients, d=12 data)')

### Variant 2: steeper data

The data are regenerated at `d = 15`. The limit cycle exists only above a steepness threshold
(a Hopf-type onset), so steeper data widen the band of parameters whose smooth model still
oscillates.

In [ ]:
STEEP_D = 15.0
ts15, xs15 = [], []
for x0 in x0s:
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(P, STEEP_D), rtol=1e-9, atol=1e-11)
    ts15.append(t); xs15.append(sol.y.T)

sweep(False, ts15, xs15, gap, d=STEEP_D, label='NAIVE  (raw gradients, d=15 data)')

## 4. PINN with Fourier feature mapping

As in Example 3, the surrogate receives `sin`/`cos` of (normalized) time. The loss carries the
same nine `(L, U, theta)` as learnable parameters (`d`, `gamma` fixed), scores both tests, and
uses the smoothed data of Variant 1.

In [ ]:
def rhs_t(s, p, d=DATA_D, g=GAMMA):
    x, y, z = s[:,0:1], s[:,1:2], s[:,2:3]
    dx = -g*x + hill_rep_t(z, p['L31'],p['U31'],p['th31'], d)
    dy = -g*y + hill_rep_t(x, p['L12'],p['U12'],p['th12'], d)
    dz = -g*z + hill_rep_t(y, p['L23'],p['U23'],p['th23'], d)
    return torch.cat([dx, dy, dz], dim=1)

def build_tensors(ts, xs, x0s):
    t_d  = np.concatenate([t[:, None] for t in ts])
    x_d  = np.vstack(xs)
    x0_d = np.vstack([np.tile(x0, (len(t), 1)) for x0, t in zip(x0s, ts)])
    t_ic = np.zeros((len(x0s), 1)); x_ic = np.array(x0s, dtype=float)
    to = lambda a: torch.tensor(a)
    return to(t_d), to(x0_d), to(x_d), to(t_ic), to(x_ic)

In [ ]:
class PINN(nn.Module):
    def __init__(self, m, param_init, T, hidden=64, depth=4, fourier_k=6):
        super().__init__()
        self.m, self.T, self.fourier_k = m, T, fourier_k
        in_dim = (2*fourier_k if fourier_k > 0 else 1) + m
        if fourier_k > 0:
            self.register_buffer('freqs', 2*np.pi*torch.arange(1, fourier_k+1).double())
        layers, d0 = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d0, hidden), nn.Tanh()]; d0 = hidden
        layers += [nn.Linear(d0, m)]
        self.net = nn.Sequential(*layers)
        # physical parameters via positive reparametrization p = raw^2 + eps
        self.raw = nn.ParameterDict(
            {k: nn.Parameter(torch.tensor(float(v)**0.5)) for k, v in param_init.items()})
    def phys_params(self):
        return {k: self.raw[k]**2 + 1e-6 for k in self.raw}
    def _feat(self, t):
        tn = t / self.T
        if self.fourier_k > 0:
            return torch.cat([torch.sin(self.freqs*tn), torch.cos(self.freqs*tn)], dim=1)
        return tn
    def forward(self, t, x0):
        return self.net(torch.cat([self._feat(t), x0], dim=1))

def fit_pinn(model, data, steps=8000, lr=1e-3, w=(50.0,10.0,1.0), log=1000):
    t_d, x0_d, x_d, t_ic, x_ic = data
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for it in range(steps):
        opt.zero_grad()
        loss_data = ((model(t_d, x0_d) - x_d)**2).mean()
        tc = t_d.clone().requires_grad_(True)
        u = model(tc, x0_d)
        grads = [torch.autograd.grad(u[:, j].sum(), tc, create_graph=True)[0] for j in range(model.m)]
        du = torch.cat(grads, dim=1)
        loss_phys = ((du - rhs_t(u, model.phys_params()))**2).mean()
        loss_ic = ((model(t_ic, x_ic) - x_ic)**2).mean()
        loss = w[0]*loss_data + w[1]*loss_phys + w[2]*loss_ic
        loss.backward(); opt.step()
        if log and it % log == 0:
            print(f'  step {it:5d}  data {loss_data.item():.2e}  phys {loss_phys.item():.2e}')
    return model

In [ ]:
# feed the PINN the lightly-smoothed data (tweak 1), neutral parameter init
xs_fit = [np.column_stack([savgol_filter(y[:,k], 21, 3) for k in range(3)]) for y in xs_noisy]
init = {k: (0.05 if k[0]=='L' else 1.0) for k in KEYS}   # L~0, U~1, theta~1
torch.manual_seed(0)
model = PINN(m=3, param_init=init, T=T, hidden=64, depth=4, fourier_k=6)
model = fit_pinn(model, build_tensors(ts, xs_fit, x0s), steps=8000, lr=1e-3, w=(50.0,10.0,1.0))

pp = {k: float(v) for k, v in model.phys_params().items()}
print('PINN estimate:', {k: round(pp[k],2) for k in KEYS})
print('  satisfies DSGRN inequalities:', in_region(pp))
print('  learned model oscillates:    ', oscillates(pp))

## 5. Conley-Morse graph (DSGRN, optional)

Installs DSGRN and computes the Morse graph of node 13 directly, rather than via the explicit
inequalities. The region's Morse graph is a single `FC` node.

In [ ]:
# Optional: building DSGRN needs a C++ toolchain (a few minutes on Colab).
# !pip -q install networkx DSGRN
try:
    import DSGRN
    spec = 'x : ~z\ny : ~x\nz : ~y\n'
    net = DSGRN.Network(spec); pg = DSGRN.ParameterGraph(net)
    region = pg.parameter(13)
    dg = DSGRN.DomainGraph(region); dec = DSGRN.MorseDecomposition(dg.digraph())
    mg = DSGRN.MorseGraph(dg, dec)
    print('parameter node 13 Morse graph:', mg.stringify())   # -> single FC node
    print('inequalities:', region.inequalities()[:88], '...')
except ImportError:
    print('DSGRN not installed - the inequality test above is the standalone substitute.')

## Morse graph and Conley-index recovery (DSGRN, optional)

Two criteria per recovered parameter set: exact DSGRN region equality, and label-preserving
isomorphism of the **Conley-Morse graph** (recurrent Morse sets, reachability order,
Conley-index labels). Region equality implies isomorphic Morse graphs, so the second criterion
is the weaker one.

The cells below build DSGRN (C++ toolchain required) and compare the recovered and target Morse
graphs via `par_index_from_sample` (parameters to region index), `DSGRN.Blowup.ConleyMorseGraph`
(region to Morse graph), and `DSGRN.isomorphic_morse_graphs` (`node_match` on the location-free
Conley index).

In [ ]:
# Optional: building DSGRN needs a C++ toolchain (a few minutes on Colab).
# !pip -q install networkx DSGRN
DSGRN_OK = False
try:
    import DSGRN, numpy as np
    NET_SPEC = 'x : ~z\ny : ~x\nz : ~y\n'
    _net = DSGRN.Network(NET_SPEC)
    _pg  = DSGRN.ParameterGraph(_net)
    TARGET = 13
    _mg = {}
    def conley_morse(idx):          # region index -> annotated Conley-Morse graph (cached)
        if idx not in _mg:
            _mg[idx] = DSGRN.Blowup.ConleyMorseGraph(_pg.parameter(idx))[0]
        return _mg[idx]
    def to_matrices(p):             # learned (L, U, theta) -> DSGRN L, U, T matrices
        n = 3
        L = np.zeros((n, n)); U = np.zeros((n, n)); T = np.zeros((n, n))
        L[2,0],U[2,0]=p['L31'],p['U31']; L[0,1],U[0,1]=p['L12'],p['U12']; L[1,2],U[1,2]=p['L23'],p['U23']
        T[2,0],T[0,1],T[1,2]=p['th31'],p['th12'],p['th23']
        return L, U, T
    def region_of(p):               # learned parameters -> DSGRN region index (-1 if outside)
        L, U, T = to_matrices(p)
        return DSGRN.par_index_from_sample(_pg, L, U, T)
    def morse_recovers(idx, target=TARGET):   # same Conley-Morse graph up to label-preserving iso
        return idx >= 0 and DSGRN.isomorphic_morse_graphs(conley_morse(idx), conley_morse(target))
    print('target region', TARGET, 'Conley indices:',
          [conley_morse(TARGET).vertex_label(v)[2] for v in conley_morse(TARGET).vertices()])
    DSGRN_OK = True
except Exception as e:
    print('DSGRN unavailable - skipping this section:', repr(e)[:90])

In [ ]:
if DSGRN_OK:
    # the recovered parameters from the cells above
    p_ls = ls_recover(ts, xs_noisy)
    print('recovered region index:', region_of(p_ls),
          '| exact region match:', region_of(p_ls) == TARGET,
          '| Morse/Conley match:', morse_recovers(region_of(p_ls)))

    # noise sweep: exact-region recovery vs Morse/Conley recovery (least squares)
    levels = [0.0, 0.1, 0.25, 0.5]
    print(f'{"noise":>6} | {"exact region":>12} | {"Morse/Conley":>12} | {"region-miss, Morse-ok":>20}')
    for ub in levels:
        r = np.random.default_rng(7); reg = mor = gapc = 0
        for _ in range(15):
            xs_n = [add_noise(y, ub, gap, r) for y in xs]
            idx = region_of(ls_recover(ts, xs_n))
            er = (idx == TARGET); mr = morse_recovers(idx)
            reg += er; mor += mr; gapc += (mr and not er)
        print(f'{ub:>6} | {reg:>10}/15 | {mor:>10}/15 | {gapc:>18}/15')
    # repressilator FC: node 13 is the unique periodic-orbit region